<a href="https://colab.research.google.com/github/garnfelfel/Deep_Learning_Homeworks/blob/main/TransformerEncoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Imports**


In [20]:
!pip uninstall numpy
!pip install --upgrade numpy==1.15.0


  Using cached numpy-1.15.0.zip (4.5 MB)
  Preparing metadata (setup.py) ... done
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for numpy
  Running setup.py clean for numpy
  error: subprocess-exited-with-error
  
  × python setup.py clean did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed cleaning build dir for numpy
Failed to build numpy
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (numpy)


In [2]:
import os
import glob
import random
import shutil
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import torch.nn.utils.rnn as rnn_utils
from google.colab import drive
import torch.nn as nn
import math
import torch.optim as optim
import jiwer
import json
import time
#from huggingface_hub import hf_hub_download
from pyctcdecode import build_ctcdecoder
import numpy as np
import csv

In [3]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
source_data_dir =  '/content/drive/MyDrive/EEG-to-Text/splits'
train_dir = os.path.join(source_data_dir, 'train')
test_dir = os.path.join(source_data_dir, 'test' )
val_dir = os.path.join(source_data_dir, 'val')
#!ls "$source_data_dir"
#print("Train data")
#!ls "$train_dir"
#print("Test data")
#!ls "$test_dir"
#print("Validation data")
#!ls "$val_dir"

# **Debugging splits**

In [4]:
# Define how many files for debugging
N_per = [5*i/100 for i in range(1,7)]
N_TRAIN = 756*N_per
N_VAL = 100*N_per
N_TEST = 100*N_per

# creating debug splits
small_splits_dir = '/content/drive/MyDrive/EEG-to-Text/small_splits'

small_train_dir = os.path.join(small_splits_dir, 'train')
small_val_dir = os.path.join(small_splits_dir, 'val')
small_test_dir = os.path.join(small_splits_dir, 'test')

#!ls "$source_data_dir"
#print("Train data")
#!ls "$small_train_dir"
#print("Test data")
#!ls "$small_test_dir"
#print("Validation data")
#!ls "$small_val_dir"

# **Step 1**

Create vocab dict

In [5]:
# Dictionaries for mapping
char_to_int = {}
int_to_char = {}
BLANK_TOKEN_INDEX = 0
# Add the CTC blank token
char_to_int['<BLANK>'] = BLANK_TOKEN_INDEX
int_to_char[BLANK_TOKEN_INDEX] = '<BLANK>'

# Define vocabulary
vocab = "abcdefghijklmnopqrstuvwxyz0123456789' "

# Add characters starting from index 1
for i, char in enumerate(vocab, start=1):
    char_to_int[char] = i
    int_to_char[i] = char

# Total number of classes
NUM_GRAPHEMES = len(char_to_int)


print("Number of graphemes:", NUM_GRAPHEMES)
print("char_to_int :", char_to_int)
print("int_to_char :", int_to_char)
print("Example mapping:", char_to_int['a'], "->", int_to_char[char_to_int['a']])

# Create the Vocabulary JSON File
LABELS_FILE_PATH = '/content/drive/MyDrive/EEG-to-Text/labels.json'
labels = [int_to_char.get(i, '') for i in range(len(int_to_char))]
labels[BLANK_TOKEN_INDEX] = ""  # Blank token must be an empty string
with open(LABELS_FILE_PATH, 'w') as f:
    json.dump(labels, f)
print(f"Created {LABELS_FILE_PATH} with {len(labels)} graphemes.")


Number of graphemes: 39
char_to_int : {'<BLANK>': 0, 'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'o': 15, 'p': 16, 'q': 17, 'r': 18, 's': 19, 't': 20, 'u': 21, 'v': 22, 'w': 23, 'x': 24, 'y': 25, 'z': 26, '0': 27, '1': 28, '2': 29, '3': 30, '4': 31, '5': 32, '6': 33, '7': 34, '8': 35, '9': 36, "'": 37, ' ': 38}
int_to_char : {0: '<BLANK>', 1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 27: '0', 28: '1', 29: '2', 30: '3', 31: '4', 32: '5', 33: '6', 34: '7', 35: '8', 36: '9', 37: "'", 38: ' '}
Example mapping: 1 -> a
Created /content/drive/MyDrive/EEG-to-Text/labels.json with 39 graphemes.


# **Step 2**
Create the PyTorch Dataset (Mini Dataset for debugging purpose)
This is the Dataset class that only loads the pre-processed .pt files.

In [6]:
class EEGWordDataset_MultiSubject(Dataset):
    """
    Loads data from multiple subjects by padding the channel dimension.
    """
    def __init__(self, data_dir, char_map, max_channels, len_data):
        self.data_dir = data_dir
        self.char_map = char_map
        self.max_channels = max_channels # Max channels found in your dataset

        self.samples = []
        for f in os.listdir(data_dir):
            if f.endswith('.pt'):
                word = f.split('_')[0]
                self.samples.append((os.path.join(self.data_dir, f), word))
        self.samples = self.samples[:len_data]
        # print(f"Loaded {len(self.samples)} samples from {data_dir}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        file_path, word = self.samples[idx]

        # Load the 3D tensor
        # Shape: [num_frames, window_size, num_channels]
        x_tensor = torch.load(file_path)

        num_frames, window_size, num_channels = x_tensor.shape

        # Pad the channel dimension if necessary
        if num_channels < self.max_channels:
            # Calculate padding needed
            pad_width = self.max_channels - num_channels

            # Pad LAST dimension (channels)
            # (pad_left, pad_right, pad_top, pad_bottom, pad_front, pad_back)
            # We only want to pad the last dim: (0, pad_width)
            x_tensor = F.pad(x_tensor, (0, pad_width), "constant", 0)

        # Flatten features
        # Shape: [num_frames, window_size * self.max_channels]
        x_tensor = x_tensor.flatten(start_dim=1)

        # Encode the target word
        target = [self.char_map[char] for char in word.lower() if char in self.char_map]
        y_tensor = torch.tensor(target, dtype=torch.long)

        return x_tensor, y_tensor

# Step 3
the Collate Function :

This function takes a list of samples from the Dataset (which all have different lengths) and bundles them into a single, padded batch.

In [7]:
def ctc_collate_fn(batch):
    """
    Processes a list of (sequence, target) tuples
    and turns them into padded batches.
    """
    # Separate sequences and targets
    sequences = [item[0] for item in batch]
    targets = [item[1] for item in batch]
    # print(sequences)
    # print(targets)
    # Get original lengths (for CTC loss)
    seq_lengths = torch.tensor([len(s) for s in sequences], dtype=torch.long)
    target_lengths = torch.tensor([len(t) for t in targets], dtype=torch.long)

    # Pad the sequences (batch_first=False for model)
    # Output shape: [SeqLen, BatchSize, FeatureDim]
    padded_seqs = rnn_utils.pad_sequence(
        sequences,
        batch_first=True,
        padding_value=0.0
    )

    # Pad the targets (batch_first=True for loss)
    # Output shape: [BatchSize, MaxTargetLen]
    padded_targets = rnn_utils.pad_sequence(
        targets,
        batch_first=True,
        padding_value=0
    )

    return padded_seqs, padded_targets, seq_lengths, target_lengths


# **Step 4** : CTC-Transformer Model

This code defines a model that uses a **TransformerEncoder** to process the HFA sequence and a final linear layer to project the output into the grapheme space, suitable for the CTC loss function.

In [8]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))

        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe)  # (max_len, d_model)

    def forward(self, x):
        """
        x: (batch, seq_len, d_model)
        """
        seq_len = x.size(1)
        x = x + self.pe[:seq_len].unsqueeze(0)  # (1, seq, d_model)
        return self.dropout(x)

class BCIToTextModel(nn.Module):
    """
    CTC-Transformer model for BCI-to-Text decoding based on.

    This model implements the "Core Architecture":
    1.  An input layer to project HFA features to the model's dimension.
    2.  Positional encoding to inject sequence information.
    3.  A Multi-Head Transformer Encoder to capture dependencies.
    4.  An output layer to map encoder hidden states to grapheme probabilities.
    """
    def __init__(self,
                 input_feat_dim: int,
                 num_graphemes: int,
                 nhead: int = 8,
                 d_model: int = 512,
                 num_encoder_layers: int = 6,
                 dim_feedforward: int = 2048,
                 dropout: float = 0.1):
        """
        Args:
            input_feat_dim (int): Dimension of the input HFA feature vector.
                                  (Derived from 1103 electrodes + windowing)
            num_graphemes (int): The number of output classes (letter, special chars + CTC blank token).
            nhead (int): Number of heads in the "Multi-Head Transformer Encoder".
            d_model (int): The hidden dimension of the transformer.
            num_encoder_layers (int): Number of layers in the encoder.
            dim_feedforward (int): Dimension of the FFN inside the transformer.
            dropout (float): Dropout rate.
        """
        super().__init__()
        self.d_model = d_model

        # Input feature projection : Mapping the high-dimensional HFA vector to the model's working dimension
        self.input_projection = nn.Linear(input_feat_dim, d_model)

        # Positional Encoding
        self.pos_encoder = PositionalEncoding(d_model, dropout)

        # CTC-Transformer (Multi-Head Transformer Encoder)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True

        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer=encoder_layer,
            num_layers=num_encoder_layers
        )

        # Output Layer
        # Projects encoder output to the grapheme vocabulary size
        self.fc_out = nn.Linear(d_model, num_graphemes)

        # LogSoftmax for CTC Loss
        # nn.CTCLoss expects log-probabilities as input
        self.log_softmax = nn.LogSoftmax(dim=2)

    def forward(self, src, src_key_padding_mask=None):
      # src should be (batch, seq, feat)
      # print("SRC entering model.forward():", src.shape)
      #if src_key_padding_mask is not None:
          #print("MASK entering model.forward():", src_key_padding_mask.shape)

      # quick sanity asserts to fail early with helpful message
      assert src.dim() == 3, f"src must be 3D (B,S,F). got {src.dim()}D"
      # check first dim of mask equals batch dim
      if src_key_padding_mask is not None:
          assert src_key_padding_mask.dim() == 2, "mask must be 2D (B,S)"
          assert src.shape[0] == src_key_padding_mask.shape[0], (
              f"Batch dim mismatch: src.shape[0]={src.shape[0]} vs mask.shape[0]={src_key_padding_mask.shape[0]}. "
              "This means src is not batch-first."
          )

      src = self.input_projection(src) * math.sqrt(self.d_model)
      src = self.pos_encoder(src)
      memory = self.transformer_encoder(
          src,
          src_key_padding_mask=src_key_padding_mask
      )
      output = self.fc_out(memory)
      return self.log_softmax(output)

# **Step 5: Set Up and Run the Training Loop**

In [9]:
# Define how many files for debugging
N_per = [i/100 for i in range(5,9,5)]
print(N_per)

[0.05]


In [12]:
# Define Paths and Parameters ---
PROCESSED_DIR = '/content/drive/MyDrive/EEG-to-Text/preprocessed_frames'
TRAIN_DIR = os.path.join(PROCESSED_DIR, 'train')
VAL_DIR = os.path.join(PROCESSED_DIR, 'val')

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_CHANNELS = 127
NUM_EPOCHS = 10
BATCH_SIZE = 16


CHECKPOINT_DIR = '/content/drive/MyDrive/EEG-to-Text/model_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"Using device: {DEVICE}")
for i in N_per :
  N_TRAIN = int(696 * i)
  N_VAL   = int(149 * i)
  N_TEST  = int(150 * i)
  def train_save_model(N_TRAIN=696,N_VAL=149, N_TEST=149, train_dir = TRAIN_DIR, val_dir = VAL_DIR, char_to_int_fct=char_to_int, max_chan = MAX_CHANNELS, ctc_collate = ctc_collate_fn, batch_size= BATCH_SIZE, num_workers = 4, num_graphemes= NUM_GRAPHEMES, n_head =8, d_model= 512, num_encoder_layers=6, dim_feedforward = 2048, num_epochs = NUM_EPOCHS):
    print(f"training with for N_train = {N_TRAIN}, N_val = {N_VAL}")
    # Instantiate Datasets
    train_dataset = EEGWordDataset_MultiSubject(
        data_dir=train_dir,
        char_map=char_to_int_fct,
        max_channels=max_chan,
        len_data=N_TRAIN
    )
    val_dataset = EEGWordDataset_MultiSubject(
        data_dir=val_dir,
        char_map=char_to_int_fct,
        max_channels=max_chan,
        len_data=N_VAL
    )

    # Instantiate DataLoaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=ctc_collate,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=ctc_collate,
        num_workers=num_workers
    )

    # Get Feature Dimension ---
    # Load one sample to find the feature dimension
    sample_seq, _ = train_dataset[0]
    HFA_FEATURE_DIM = sample_seq.shape[1] # [SeqLen, FeatureDim] -> get FeatureDim
    # print(f"Detected Feature Dimension: {HFA_FEATURE_DIM}")

    # Instantiate Model, Loss, and Optimizer ---
    model = BCIToTextModel(
        input_feat_dim=HFA_FEATURE_DIM,
        num_graphemes=num_graphemes,
        nhead=n_head,
        d_model=d_model,
        num_encoder_layers=num_encoder_layers,
        dim_feedforward= dim_feedforward
    ).to(DEVICE)

    # CTCLoss expects the 'blank' token index
    ctc_loss = nn.CTCLoss(blank=BLANK_TOKEN_INDEX)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    # The Training Loop ---

    model.train() # Set model to training mode

    for epoch in range(num_epochs):
        # print(f"\n--- Epoch {epoch+1}/{NUM_EPOCHS} ---")
        epoch_loss = 0

        for i, batch in enumerate(train_loader):
            # Move data to the GPU
            padded_seqs, padded_targets, seq_lengths, target_lengths = [
                d.to(DEVICE) for d in batch
            ]

            # --- Create padding mask for the Transformer Encoder ---
            # This tells the Transformer to ignore padded time steps
            # Shape: [BatchSize, SeqLen]
            max_seq_len = padded_seqs.shape[0]
            padding_mask = (torch.arange(max_seq_len, device=DEVICE)
                            .expand(len(seq_lengths), max_seq_len) >= seq_lengths.unsqueeze(1))

            # --- Forward pass ---
            optimizer.zero_grad()
            log_probs = model(padded_seqs, src_key_padding_mask=padding_mask)

            # Calculate Loss ---
            # CTCLoss requires log_probs in [SeqLen, BatchSize, Classes]
            loss = ctc_loss(
                log_probs,
                padded_targets,
                seq_lengths,
                target_lengths
            )

            # --- Backward pass ---
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            if (i + 1) % 10 == 0:
                print(f"  Batch {i+1}/{len(train_loader)}, Loss: {loss.item():.4f}")


        print(f"End of Epoch {epoch+1}, Average Loss: {epoch_loss / len(train_loader):.4f}")
    # --- MODIFIED: Model Saving Logic Added Here ---
    # Save checkpoint after each epoch for later WER evaluation
    percentage_str = f"{N_TRAIN*100/696}"
    checkpoint_path = os.path.join(
        CHECKPOINT_DIR,
        f"model_pct{percentage_str}_epoch{epoch+1}.pth"
    )
    torch.save(model.state_dict(), checkpoint_path)
    print(f"Model checkpoint saved to {checkpoint_path}")
    # --- END MODIFIED ---

    print(f"Training Complete \n")


Using device: cpu


# **Linguistic Refinement**

In [ ]:
print("/nPreparing Decoder ---")
!pip install https://github.com/kpu/kenlm/archive/master.zip

# Download the Language Model ---
repo_id = "BramVanroy/kenlm_wikipedia_nl"
filename = "wiki_nl_token.arpa.bin"
print(f"Downloading {filename} from {repo_id}...")
LANGUAGE_MODEL_FILE = ""
try:
    LANGUAGE_MODEL_FILE = hf_hub_download(
        repo_id=repo_id,
        filename=filename
    )
    print(f"Language Model saved to: {LANGUAGE_MODEL_FILE}")
except Exception as e:
    print(f"Error downloading LM: {e}")
    print("Please check your internet connection or the Hugging Face repo.")

/nPreparing Decoder ---


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


wiki_nl_token.arpa.bin:   0%|          | 0.00/14.0G [00:00<?, ?B/s]

Language Model saved to: /root/.cache/huggingface/hub/models--BramVanroy--kenlm_wikipedia_nl/snapshots/0b9a2ce5bbec0b16486808182b79c9d0dc2a5e28/wiki_nl_token.arpa.bin
Created /content/drive/MyDrive/EEG-to-Text/labels.json with 39 graphemes.


In [ ]:
target_path = "/content/drive/MyDrive/EEG-to-Text/wiki_nl_token.arpa.bin"
shutil.copy(LANGUAGE_MODEL_FILE, target_path)

'/content/drive/MyDrive/EEG-to-Text/wiki_nl_token.arpa.bin'

In [10]:
import math
import kenlm

lm_path = "/content/drive/MyDrive/EEG-to-Text/wiki_nl_token.arpa.bin"
lm = kenlm.Model(lm_path)

def logP_lm_word(word: str) -> float:
    # KenLM returns log10 probability
    log10_p = lm.score(word, bos=True, eos=True)
    return log10_p * math.log(10.0)
print("Completed")

In [11]:
# Load the CTC Beam Search Decoder
try:
    decoder = build_ctcdecoder(
        labels,
        kenlm_model_path=lm_path,
        alpha=1.0,  # How much to trust the LM
        beta=0.5,   # How much to reward word length
    )
    print("Successfully loaded CTC Beam Search Decoder.")
except Exception as e:
    print(f"Error loading Decoder: {e}")

Successfully loaded CTC Beam Search Decoder.


In [17]:
def decode_target_batch(target_ids, int_to_char, pad_index=0):
    """Convert padded target indices to strings."""
    decoded_batch = []
    for seq in target_ids:
        chars = [int_to_char.get(idx.item(), '?')
                 for idx in seq
                 if idx.item() != pad_index]
        decoded_batch.append("".join(chars))
    return decoded_batch


def greedy_decode_batch(log_probs, int_to_char, blank_index):
    """
    log_probs must be (T, B, C).
    Greedy CTC decoding with collapsing repeats and removing blanks.
    """
    T, B, C = log_probs.shape
    best_path = torch.argmax(log_probs, dim=2)  # (T, B)

    decoded_batch = []

    for b in range(B):
        seq = best_path[:, b]
        decoded_word = []
        last = -1
        for idx in seq:
            idx = idx.item()
            if idx == last:
                continue
            if idx == blank_index:
                last = idx
                continue
            decoded_word.append(int_to_char.get(idx, '?'))
            last = idx
        decoded_batch.append("".join(decoded_word))

    return decoded_batch


def beam_search_decode_batch(log_probs, decoder, expect_log_probs=True):
    """
    LM-based CTC decoding.
    decoder.decode expects shape (T, C) per sample.
    """

    T, B, C = log_probs.shape
    print("T:", T)
    print("B:", B)
    print("C:", C)
    decoded_batch = []
    for b in range(B):
        lp = log_probs[:, b, :]
        print("lp: ", lp)
        # LM usually expects probabilities
        if expect_log_probs:
            t_probs = lp.exp()
            print("t_probs: ", t_probs)
            cpu_probs = t_probs.cpu()
            print("cpu_probs: ", cpu_probs)
            probs = cpu_probs.numpy()
        else:
            probs = lp.cpu().numpy()
            print("probs: ", probs)

        text = decoder.decode(probs)
        print("text: ", text)
        decoded_batch.append(text)
        print("decoded_batch: ", decoded_batch)

    return decoded_batch


In [14]:
# --- Paths & params ---
PROCESSED_DIR = '/content/drive/MyDrive/EEG-to-Text/preprocessed_frames'
TRAIN_DIR = os.path.join(PROCESSED_DIR, 'train')
VAL_DIR = os.path.join(PROCESSED_DIR, 'val')

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_CHANNELS = 127
NUM_EPOCHS = 10
BATCH_SIZE = 16

CHECKPOINT_DIR = '/content/drive/MyDrive/EEG-to-Text/Fixed_model_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
METRICS_CSV = os.path.join(CHECKPOINT_DIR, "evaluation_metrics.csv")

# Create CSV with header if missing
if not os.path.exists(METRICS_CSV):
    with open(METRICS_CSV, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([
            "percentage",
            "model_name",
            "epoch",
            "N_TRAIN",
            "N_VAL",
            "greedy_WER",
            "LM_WER",
            "LM_improvement"
        ])

print(f"Using device: {DEVICE}")
N_per = [0.1]
# N_per should be a list of percentages in decimal (e.g. [0.05, 0.10, ...])
# Ensure N_per is defined before this script
for pct in N_per:
    N_TRAIN = round(696 * pct)
    N_VAL   = round(149 * pct)
    N_TEST  = round(150 * pct)
    percentage_str = f"{round(N_TRAIN*100/696)}"
    print(f"\n---------------- for {percentage_str}% : N_train = {N_TRAIN}, N_val = {N_VAL}")

    # --- Datasets ---
    train_dataset = EEGWordDataset_MultiSubject(
        data_dir=TRAIN_DIR,
        char_map=char_to_int,
        max_channels=MAX_CHANNELS,
        len_data=N_TRAIN
    )
    val_dataset = EEGWordDataset_MultiSubject(
        data_dir=VAL_DIR,
        char_map=char_to_int,
        max_channels=MAX_CHANNELS,
        len_data=N_VAL
    )

    # --- DataLoaders (no multiprocessing in notebooks/Colab) ---
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=ctc_collate_fn,
        num_workers=0
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=ctc_collate_fn,
        num_workers=0
    )

    # --- Feature dim discovery ---
    sample_seq, _ = train_dataset[0]
    HFA_FEATURE_DIM = sample_seq.shape[1]  # [SeqLen, FeatureDim]

    # --- Model / loss / optimizer ---
    model = BCIToTextModel(
        input_feat_dim=HFA_FEATURE_DIM,
        num_graphemes=NUM_GRAPHEMES,
        nhead=8,
        d_model=512,
        num_encoder_layers=6,
        dim_feedforward=2048
    ).to(DEVICE)

    # make sure model returns log-probs in shape [SeqLen, Batch, Classes] for CTCLoss
    ctc_loss = nn.CTCLoss(blank=BLANK_TOKEN_INDEX)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    # --- Training ---
    model.train()

    for epoch in range(NUM_EPOCHS):
        epoch_loss = 0.0
        for batch_idx, batch in enumerate(train_loader):  # renamed idx to avoid shadowing
            padded_seqs, padded_targets, seq_lengths, target_lengths = [d.to(DEVICE) for d in batch]

            # padding mask: SeqLen is dim 1 (Batch, SeqLen, FeatureDim)
            max_seq_len = padded_seqs.shape[1]
            padding_mask = (torch.arange(max_seq_len, device=DEVICE)
                            .expand(len(seq_lengths), max_seq_len) >= seq_lengths.unsqueeze(1))
            #print("BATCH padded_seqs.shape (from dataloader):", padded_seqs.shape)   # expect (B, S, F)
            #print("BATCH padding_mask.shape (constructed):", padding_mask.shape)     # expect (B, S)
            #print("padding_mask dtype:", padding_mask.dtype)
            optimizer.zero_grad()
            log_probs = model(padded_seqs, src_key_padding_mask=padding_mask)  # ensure model expects this
            log_probs = log_probs.permute(1, 0, 2)
            # CTCLoss expects: (T, N, C) for inputs -> if model returns (T,N,C) ok, otherwise transpose
            # If model returns (N, T, C) -> do log_probs = log_probs.permute(1, 0, 2)
            # Ensure dtype: log_probs should be log probabilities (log_softmax)
            loss = ctc_loss(
                log_probs,
                padded_targets,
                seq_lengths,
                target_lengths
            )

            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            if (batch_idx + 1) % 10 == 0:
                print(f"  Batch {batch_idx+1}/{len(train_loader)}, Loss: {loss.item():.4f}")

        avg_loss = epoch_loss / max(1, len(train_loader))
        print(f"End of Epoch {epoch+1}, Average Loss: {avg_loss:.4f}")

    # --- Save final checkpoint for this percentage (epoch is last epoch index) ---
    model_name = f"model_pct{percentage_str}_epoch{NUM_EPOCHS}.pth"
    checkpoint_path = os.path.join(CHECKPOINT_DIR, model_name)
    torch.save(model.state_dict(), checkpoint_path)
    print(f"Model checkpoint saved to {checkpoint_path}")
    print(f"Training Complete for percentage {percentage_str} % \n")

    # --- Evaluation ---
    print(f"Starting Model Evaluation for percentage {percentage_str} % \n")
    model.eval()
    model.to(DEVICE)

    all_targets = []
    all_greedy_preds = []
    all_lm_preds = []

    start_time = time.time()
    with torch.no_grad():
        for x, batch in enumerate(val_loader):
            padded_seqs, padded_targets, seq_lengths, target_lengths = [d.to(DEVICE) for d in batch]
            max_seq_len = padded_seqs.shape[1]
            padding_mask = (torch.arange(max_seq_len, device=DEVICE)
                            .expand(len(seq_lengths), max_seq_len) >= seq_lengths.unsqueeze(1))

            log_probs = model(padded_seqs, src_key_padding_mask=padding_mask)
            log_probs_for_decode = log_probs.permute(1, 0, 2).cpu()
            print("During eval log_probs: ", log_probs_for_decode)
            print("During eval log_probs shape:", log_probs.shape)

            truth_text = decode_target_batch(padded_targets.cpu(), int_to_char)
            print("truth_text:" , truth_text)
            greedy_text = greedy_decode_batch(log_probs_for_decode, int_to_char, BLANK_TOKEN_INDEX)
            print("greedy_text:" , greedy_text)
            lm_text = beam_search_decode_batch(log_probs_for_decode, decoder)
            print("lm_text:" , lm_text)

            all_targets.extend(truth_text)
            all_greedy_preds.extend(greedy_text)
            all_lm_preds.extend(lm_text)

            if (x + 1) % 10 == 0:
                print(f"  Processed batch {x+1} / {len(val_loader)}")

    end_time = time.time()
    print(f"Evaluation Complete ({end_time - start_time:.2f}s)")

    greedy_wer = jiwer.wer(all_targets, all_greedy_preds)
    lm_wer = jiwer.wer(all_targets, all_lm_preds)
    improvement = greedy_wer - lm_wer

    print("\nPerformance Results ")
    print(f"  Greedy Decoding (No LM) WER:     {greedy_wer * 100:.2f}%")
    print(f"  Linguistic Refinement (LM) WER:  {lm_wer * 100:.2f}%")
    print("-" * 30)
    print(f"  Improvement from LM:             {improvement * 100:.2f}%")

    print("\nExample Decodings")
    print("TARGET".ljust(15), "| GREEDY".ljust(15), "| LM REFINED")
    print("-" * 45)
    for z in range(min(20, len(all_targets))):
        print(f"{all_targets[z]: <15} | {all_greedy_preds[z]: <15} | {all_lm_preds[z]}")
    print(f"Evaluation complete for {percentage_str}")

    # --- Save metrics to CSV immediately for this percentage ---
    with open(METRICS_CSV, 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([
            percentage_str,
            model_name,
            NUM_EPOCHS,
            N_TRAIN,
            N_VAL,
            greedy_wer,
            lm_wer,
            improvement
        ])

    print(f"Saved metrics for {percentage_str}% → {METRICS_CSV}")


Using device: cpu

---------------- for 10% : N_train = 70, N_val = 15
End of Epoch 1, Average Loss: 88.6615
End of Epoch 2, Average Loss: 5.0896
End of Epoch 3, Average Loss: 5.4531
End of Epoch 4, Average Loss: 4.9964
End of Epoch 5, Average Loss: 4.2350
End of Epoch 6, Average Loss: 4.1713
End of Epoch 7, Average Loss: 4.0689
End of Epoch 8, Average Loss: 3.9770
End of Epoch 9, Average Loss: 3.9629
End of Epoch 10, Average Loss: 3.8363
Model checkpoint saved to /content/drive/MyDrive/EEG-to-Text/Fixed_model_checkpoints/model_pct10_epoch10.pth
Training Complete for percentage 10 % 

Starting Model Evaluation for percentage 10 % 



/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:515: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


During eval log_probs shape: torch.Size([15, 200, 39])
T: 200
B: 15
C: 39
lp:  tensor([[-0.0170, -7.5265, -8.2282,  ..., -8.2946, -8.5921, -8.9198],
        [-0.0169, -7.5291, -8.2393,  ..., -8.3062, -8.5865, -8.9275],
        [-0.0172, -7.5050, -8.2066,  ..., -8.2648, -8.5764, -8.9069],
        ...,
        [-0.0182, -7.4488, -8.1283,  ..., -8.2607, -8.5205, -8.8090],
        [-0.0179, -7.4409, -8.1414,  ..., -8.3154, -8.5389, -8.8197],
        [-0.0180, -7.4763, -8.1258,  ..., -8.2168, -8.5311, -8.8153]])
t_probs:  tensor([[9.8317e-01, 5.3859e-04, 2.6702e-04,  ..., 2.4987e-04, 1.8556e-04,
         1.3372e-04],
        [9.8328e-01, 5.3724e-04, 2.6407e-04,  ..., 2.4697e-04, 1.8660e-04,
         1.3269e-04],
        [9.8294e-01, 5.5032e-04, 2.7286e-04,  ..., 2.5742e-04, 1.8850e-04,
         1.3545e-04],
        ...,
        [9.8198e-01, 5.8215e-04, 2.9508e-04,  ..., 2.5849e-04, 1.9935e-04,
         1.4938e-04],
        [9.8222e-01, 5.8673e-04, 2.9123e-04,  ..., 2.4471e-04, 1.9571e-04,
 

RuntimeError: Unable to configure default ndarray.__str__

In [23]:
# --- Evaluation ---
print(f"Starting Model Evaluation for percentage {percentage_str} % \n")
model.eval()
model.to(DEVICE)

all_targets = []
all_greedy_preds = []
all_lm_preds = []

start_time = time.time()
with torch.no_grad():
    for x, batch in enumerate(val_loader):
        padded_seqs, padded_targets, seq_lengths, target_lengths = [d.to(DEVICE) for d in batch]
        max_seq_len = padded_seqs.shape[1]
        padding_mask = (torch.arange(max_seq_len, device=DEVICE)
                        .expand(len(seq_lengths), max_seq_len) >= seq_lengths.unsqueeze(1))

        log_probs = model(padded_seqs, src_key_padding_mask=padding_mask)
        log_probs_for_decode = log_probs.permute(1, 0, 2).cpu()
        print("During eval log_probs: ", log_probs_for_decode)
        print("During eval log_probs shape:", log_probs.shape)

        truth_text = decode_target_batch(padded_targets.cpu(), int_to_char)
        print("truth_text:" , truth_text)
        greedy_text = greedy_decode_batch(log_probs_for_decode, int_to_char, BLANK_TOKEN_INDEX)
        print("greedy_text:" , greedy_text)
        lm_text = beam_search_decode_batch(log_probs_for_decode, decoder)
        print("lm_text:" , lm_text)

        all_targets.extend(truth_text)
        all_greedy_preds.extend(greedy_text)
        all_lm_preds.extend(lm_text)

        if (x + 1) % 10 == 0:
            print(f"  Processed batch {x+1} / {len(val_loader)}")

end_time = time.time()
print(f"Evaluation Complete ({end_time - start_time:.2f}s)")

greedy_wer = jiwer.wer(all_targets, all_greedy_preds)
lm_wer = jiwer.wer(all_targets, all_lm_preds)
improvement = greedy_wer - lm_wer

print("\nPerformance Results ")
print(f"  Greedy Decoding (No LM) WER:     {greedy_wer * 100:.2f}%")
print(f"  Linguistic Refinement (LM) WER:  {lm_wer * 100:.2f}%")
print("-" * 30)
print(f"  Improvement from LM:             {improvement * 100:.2f}%")

print("\nExample Decodings")
print("TARGET".ljust(15), "| GREEDY".ljust(15), "| LM REFINED")
print("-" * 45)
for z in range(min(20, len(all_targets))):
    print(f"{all_targets[z]: <15} | {all_greedy_preds[z]: <15} | {all_lm_preds[z]}")
print(f"Evaluation complete for {percentage_str}")

# --- Save metrics to CSV immediately for this percentage ---
with open(METRICS_CSV, 'a', newline='') as f:
    writer = csv.writer(f)
    writer.writerow([
        percentage_str,
        model_name,
        NUM_EPOCHS,
        N_TRAIN,
        N_VAL,
        greedy_wer,
        lm_wer,
        improvement
    ])

print(f"Saved metrics for {percentage_str}% → {METRICS_CSV}")


Starting Model Evaluation for percentage 10 % 

During eval log_probs:  tensor([[[-0.0170, -7.5265, -8.2282,  ..., -8.2946, -8.5921, -8.9198],
         [-0.0170, -7.5265, -8.2282,  ..., -8.2946, -8.5921, -8.9198],
         [-0.0170, -7.5265, -8.2282,  ..., -8.2946, -8.5921, -8.9198],
         ...,
         [-0.0170, -7.5265, -8.2282,  ..., -8.2946, -8.5921, -8.9198],
         [-0.0170, -7.5265, -8.2282,  ..., -8.2946, -8.5921, -8.9198],
         [-0.0170, -7.5265, -8.2282,  ..., -8.2946, -8.5921, -8.9198]],

        [[-0.0169, -7.5291, -8.2393,  ..., -8.3062, -8.5865, -8.9275],
         [-0.0169, -7.5291, -8.2393,  ..., -8.3062, -8.5865, -8.9275],
         [-0.0169, -7.5291, -8.2393,  ..., -8.3062, -8.5865, -8.9275],
         ...,
         [-0.0169, -7.5291, -8.2393,  ..., -8.3062, -8.5865, -8.9275],
         [-0.0169, -7.5291, -8.2393,  ..., -8.3062, -8.5865, -8.9275],
         [-0.0169, -7.5291, -8.2393,  ..., -8.3062, -8.5865, -8.9275]],

        [[-0.0172, -7.5050, -8.2066,  ..., -

AttributeError: module 'numpy.core.multiarray' has no attribute 'generic'